# Clase 116 — Regularización: L1/L2, dropout, max-norm, MC dropout

Técnicas para combatir el overfitting: penalizaciones **L1/L2**, **Dropout**, **MaxNorm**, **MC Dropout** (incertidumbre) y **Stochastic Depth** (redes residuales profundas).

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`, `matplotlib`.

## 1. Penalizaciones L1 / L2 / L1_L2

`l2` (weight decay) mantiene pesos chicos; `l1` promueve sparsity. Se pasan como `kernel_regularizer`.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

reg_l2 = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(512, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.Dense(10,  activation="softmax"),
])
reg_l2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("l1:", keras.regularizers.l1(1e-4),
      "| l1_l2:", keras.regularizers.l1_l2(l1=1e-5, l2=1e-4))

## 2. Dropout

Enmascara una fracción `r` de activaciones por batch en training; en inference se desactiva (Keras escala automáticamente).

In [ ]:
con_dropout = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(512, activation="relu", kernel_initializer="he_normal"),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu", kernel_initializer="he_normal"),
    layers.Dropout(0.3),
    layers.Dense(10,  activation="softmax"),
])
con_dropout.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Dropout(0.3) entre capas; activo solo en training")

## 3. Max-norm constraint

Reescala `||w|| ≤ c` por neurona tras cada update (`kernel_constraint`).

In [ ]:
restringido = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(256, activation="relu", kernel_initializer="he_normal",
                 kernel_constraint=keras.constraints.MaxNorm(max_value=3.0)),
    layers.Dense(10, activation="softmax"),
])
print("MaxNorm(3.0) limita la norma de los pesos por unidad tras cada actualización")

## 4. Monte Carlo Dropout: incertidumbre

Con `training=True` en inference, N pasadas dan predicciones distintas → media ± std estiman la incertidumbre (Gal & Ghahramani 2016).

In [ ]:
x = tf.constant(np.random.default_rng(0).normal(size=(1, 784)), dtype=tf.float32)
predicciones = np.stack([con_dropout(x, training=True).numpy()[0] for _ in range(100)])
media = predicciones.mean(axis=0)
incertidumbre = predicciones.std(axis=0)
print("clase predicha:", int(media.argmax()))
print("std (incertidumbre) de la clase top:", round(float(incertidumbre[media.argmax()]), 4))

## 5. Stochastic Depth en un mini-ResNet

Dropear bloques residuales enteros al azar en training, con probabilidad **lineal** `p_i = i/N · p_max`.

In [ ]:
class StochasticDepth(layers.Layer):
    # Dropea el bloque residual entero con prob. drop_rate durante training.
    def __init__(self, drop_rate, **kwargs):
        super().__init__(**kwargs)
        self.drop_rate = drop_rate

    def call(self, x, training=None):
        if not training or self.drop_rate == 0.0:
            return x
        keep = tf.cast(tf.random.uniform([]) >= self.drop_rate, x.dtype)
        return keep * x / (1.0 - self.drop_rate)          # inverted scaling

def bloque_residual(x, unidades, drop_rate):
    y = layers.Dense(unidades, activation="relu", kernel_initializer="he_normal")(x)
    y = layers.Dense(unidades, kernel_initializer="he_normal")(y)
    y = StochasticDepth(drop_rate)(y)
    return layers.Activation("relu")(layers.Add()([x, y]))

inp = keras.Input(shape=(128,))
x = layers.Dense(128)(inp)
n_bloques = 8
for i in range(n_bloques):
    x = bloque_residual(x, 128, drop_rate=i / n_bloques * 0.2)   # lineal 0 -> 0.2
salida = layers.Dense(10, activation="softmax")(x)
resnet = keras.Model(inp, salida)
print("mini-ResNet con Stochastic Depth lineal:", resnet.count_params(), "params")

## Ejercicios

1. **Baseline sin regularización**: entrená un MLP grande y observá el gap train/val (≥ 5 pp).
2. **L2 y Dropout**: agregá `l2(1e-3)` y luego `Dropout(0.3)`; compará el gap.
3. **MC Dropout**: 100 predicciones con `training=True` sobre imágenes ambiguas; reportá media ± std.
4. **Stochastic Depth**: mini-ResNet de 8 bloques con `p` lineal; compará contra sin stochastic depth.

## Conclusiones

- **L2** (weight decay) mantiene pesos chicos; **L1** promueve sparsity; `λ` típico 1e-4 a 1e-3.
- **Dropout** fuerza redundancia; activo solo en training (Keras escala en inference).
- **MaxNorm** acota la norma de los pesos por unidad.
- **MC Dropout** aproxima incertidumbre bayesiana: mayor std en muestras ambiguas.
- **Stochastic Depth** (con `p` lineal) regulariza redes residuales muy profundas (ResNet, ViT vía DropPath).
- Evitá doble penalización: AdamW(wd=...) **o** `kernel_regularizer` L2, no ambos.